# Topic 4 - Full Fine-Tuning + Catastrophic Forgetting

Barclays Customer Support Intelligence System

## What you will build

In Topic 3 you used pre-trained models off the shelf, zero training, pure inference.
Now you go one step further: you fine-tune distilbert-base-uncased on Barclays complaint
data and observe both the power (higher accuracy) and the cost (catastrophic forgetting).

In Topic 3 you produced a labelled complaint dataset and saved it to S3; we load it now and fine-tune DistilBERT on it, then save the trained model's S3 location for Topic 5.

## Why this matters to Barclays

The complaints intelligence system needs to route incoming messages into the four
Barclays categories: fraud and security, billing and charges, account access, and
general enquiry. A general-purpose model gets us 55% accuracy on this 4-class task.
Fine-tuning on our own data gets us to 91%. But there is a catch: the model forgets
general language knowledge in the process.
Understanding that tradeoff is what separates a production ML engineer from a notebook hobbyist.

## Learning objectives

1. Explain when it makes economic sense to fine-tune vs use a pre-trained model as-is
2. Calculate the memory cost of full fine-tuning vs inference-only
3. Run a full fine-tuning job with HuggingFace Trainer and interpret training metrics
4. Demonstrate catastrophic forgetting by measuring GLUE benchmark collapse after fine-tuning
5. Launch a GPU training job on SageMaker with the HuggingFace estimator
6. Compare single-task vs multitask fine-tuning as a mitigation strategy

## Estimated time

80 to 100 minutes in class.


In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup for SageMaker Studio.
# All local cells run in the Studio kernel (CPU is fine for demos and dataset prep).
# The heavy fine-tuning job runs as a remote GPU job in Section 4.

!pip install -q "sagemaker>=2.200.0,<3.0.0" \
    "transformers>=4.53,<4.54" \
    "accelerate>=1.0.0" \
    "tokenizers>=0.21,<0.22" \
    "datasets>=2.18.0,<3.0.0" \
    "numpy<2"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
import sagemaker
from sagemaker import get_execution_role
import boto3
import warnings
warnings.filterwarnings("ignore")

sess   = sagemaker.Session()
role   = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")


## Where This Topic Fits

We are building the Barclays Customer Support Intelligence System end to end.
Each topic adds one layer. Today you are here:

| Step | Topic | What it adds to the system |
|------|-------|---------------------------|
| 1 | Topic 3 HuggingFace | Load pre-trained models from the Hub |
| 2 | Topic 4 Full Fine-Tuning | Adapt a model to Barclays complaints (YOU ARE HERE) |
| 3 | Topic 5 Transfer Learning | Freeze the encoder, train only the head |
| 4 | Topic 6 PEFT and LoRA | Apply the PEFT library to a full classifier |
| 5 | Topic 7 Quantization | Compress and deploy the model |

By the end of the required path you will have a fine-tuned, PEFT-adapted DistilBERT
complaint classifier compressed and running as a SageMaker endpoint.

In [ ]:
import numpy as np
import torch
import random
import os

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# CPU for all demos in this notebook; GPU job is in scripts_topic4/train.py.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")

# Sample complaint texts defined locally for this notebook's demos.
COMPLAINT_TEXTS = [
    "Unauthorised charge appeared on my account and no one is helping.",
    "I have been waiting 3 weeks for a refund with no update.",
    "The agent resolved my issue quickly, very happy with the service.",
    "My card was blocked without warning and I cannot access my funds.",
    "Great resolution, the fraud alert was handled professionally.",
]
print(f"\nSample complaint texts loaded: {len(COMPLAINT_TEXTS)}")


In [ ]:
# Handoff from Topic 3: load the labelled Barclays complaint dataset.
# In Topic 3 you routed complaints with pretrained models and saved a labelled
# dataset to S3. We load it now and extend the system: this topic fine-tunes
# DistilBERT on that exact dataset.

# --- S3 handoff helpers ---
import json, boto3, botocore

COURSE_PREFIX = "barclays-course"

def handoff_read(bucket, topic_n, artifact):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    try:
        body = boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()
        print(f"Handoff loaded: s3://{bucket}/{key}")
        return json.loads(body)
    except botocore.exceptions.ClientError:
        print(f"No handoff found at s3://{bucket}/{key} (starting mid-course is fine).")
        return None

_t3 = handoff_read(bucket, 3, "labelled_dataset.json")

if _t3 is not None:
    course_examples = _t3["examples"]
    label_map = {int(k): v for k, v in _t3["label_map"].items()}
    print(f"Loaded {len(course_examples)} labelled examples and "
          f"{len(label_map)} labels from Topic 3.")
else:
    # Fallback: student is starting at Topic 4. Define a minimal labelled set.
    print("Using a local fallback labelled dataset so Topic 4 runs standalone.")
    label_map = {0: "fraud and security", 1: "billing and charges",
                 2: "account access", 3: "general enquiry"}
    course_examples = [
        {"text": "Unauthorised charge appeared on my account.", "label": 0},
        {"text": "You charged me an overdraft fee I did not expect.", "label": 1},
        {"text": "I cannot log in to the mobile app.", "label": 2},
        {"text": "What are your branch opening hours?", "label": 3},
    ]

num_labels = len(label_map)
print(f"Fine-tuning target: {num_labels}-class Barclays complaint classifier.")


## CUDA Health Check

This kernel may be running on a GPU instance or a CPU one depending on quota.
The cell below checks whether a GPU is present and whether `torch` can use it.
If a GPU is present but `torch` is a CPU-only build, it reinstalls a matching
CUDA wheel. If you see a reinstall message, restart the kernel before continuing.


In [ ]:
# CUDA health check + safety-net.
# A SageMaker kernel may land on a GPU instance (ml.g4dn.xlarge) or a CPU one
# (ml.t3.medium) depending on quota. On a GPU box the image's torch is sometimes
# CPU-only, so torch.cuda.is_available() is wrongly False. This cell detects the
# real situation from two signals (nvidia-smi + the SageMaker metadata file) and
# reinstalls a matching CUDA wheel only when needed.
import json
import os
import re
import shutil
import subprocess
import sys

import torch


def _sagemaker_image():
    # SageMaker writes the running image/instance here. Diagnostic only.
    path = "/opt/ml/metadata/resource-metadata.json"
    if not os.path.exists(path):
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def _nvidia_smi():
    # Returns (gpu_present, driver_cuda_version_str_or_None).
    # nvidia-smi exit 0 means GPU hardware is here even if torch cannot see it.
    if shutil.which("nvidia-smi") is None:
        return False, None
    try:
        out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15)
    except Exception:
        return False, None
    if out.returncode != 0:
        return False, None
    m = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", out.stdout)
    return True, (m.group(1) if m else None)


def _pick_wheel(driver_cuda):
    # Newest cuXXX wheel index <= the driver's max CUDA.
    # SageMaker g4dn (T4) images historically ship CUDA 12.1-12.4 drivers.
    tiers = [("12.6", "cu126"), ("12.4", "cu124"), ("12.1", "cu121")]
    if not driver_cuda:
        return "cu121"  # safe default when the driver version is unreadable
    try:
        dv = tuple(int(x) for x in driver_cuda.split("."))
    except ValueError:
        return "cu121"
    for ver, tag in tiers:
        if tuple(int(x) for x in ver.split(".")) <= dv:
            return tag
    return "cu121"


meta = _sagemaker_image()
if meta:
    print("SageMaker kernel:", meta.get("AppType", "?"),
          "| instance:", meta.get("ResourceArn", "?").split("/")[-1])

has_gpu, driver_cuda = _nvidia_smi()

if not has_gpu:
    print("No GPU detected. Running on CPU kernel. No action needed.")
elif torch.version.cuda is not None:
    print(f"GPU OK. torch {torch.__version__} built for CUDA {torch.version.cuda}.")
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
else:
    wheel = _pick_wheel(driver_cuda)
    print(f"GPU present (driver CUDA {driver_cuda}) but torch is CPU-only.")
    print(f"Reinstalling torch from the {wheel} wheel index...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "torch", "torchvision", "torchaudio",
         "--index-url", f"https://download.pytorch.org/whl/{wheel}"],
        check=True,
    )
    print("Reinstalled torch with CUDA support.")
    print("RESTART THE KERNEL now, then re-run the notebook from the top.")


## Beat 1 -- Section 1 - When to Fine-Tune, When Not to Fine-Tune


A pre-trained model (Topic 3) is already very capable. So why would you ever fine-tune?

The answer is domain shift: general-purpose models are trained on the internet.
Your data is Barclays customer complaints, a specific dialect of financial English
with terms like "unauthorised charge", "direct debit reversal", "Chaps payment failure".
A model that has never seen these patterns cannot classify them reliably.

But fine-tuning has real costs:

| Concern           | Fine-Tune  | Inference Only |
|-------------------|------------|----------------|
| Accuracy (domain) | High       | Medium         |
| Memory (GPU)      | 6x model   | 1x model       |
| Training time     | Hours      | None           |
| Risk              | Forgetting | None           |
| Cost              | $10-100+   | $0.01/request  |

The rule of thumb for production:
- If a pre-trained model scores above 80% on your validation set: do NOT fine-tune yet.
  Try prompt engineering first (cheaper, faster, reversible).
- If the domain vocabulary is highly specialised (finance, law, medicine) and
  the task requires above 85% accuracy: fine-tuning is worth it.
- If you fine-tune: always measure what the model forgets (Section 3).


In [ ]:
# Beat 1: Full fine-tuning on CPU looks simple, but the memory cost is brutal.
# We calculate the memory requirement and try to run Trainer on a tiny batch.

from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
# num_labels comes from the Topic 3 label_map loaded in the handoff cell: the
# four Barclays routing categories. Never hardcode it.
model_demo = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)

# Count parameters
total_params = sum(p.numel() for p in model_demo.parameters())
trainable_params = sum(p.numel() for p in model_demo.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

# Memory for full fine-tuning with Adam optimizer (4 tensors per parameter)
# Weights: 4 bytes, Gradients: 4 bytes, Adam m: 4 bytes, Adam v: 4 bytes = 16 bytes
# Plus forward activations: roughly 2x model weights for typical batch
weight_mb    = trainable_params * 4 / 1e6
gradient_mb  = trainable_params * 4 / 1e6
optimizer_mb = trainable_params * 8 / 1e6
activations_mb = weight_mb * 2
total_mb     = weight_mb + gradient_mb + optimizer_mb + activations_mb

print(f"--- Memory cost breakdown (full fine-tuning, fp32) ---")
print(f"  Model weights:   {weight_mb:.1f} MB")
print(f"  Gradients:       {gradient_mb:.1f} MB")
print(f"  Adam optimizer:  {optimizer_mb:.1f} MB")
print(f"  Activations:     {activations_mb:.1f} MB (est)")
print(f"  TOTAL:           {total_mb:.1f} MB")
print()
print(f"Inference only:  {weight_mb:.1f} MB")
print(f"Fine-tuning:     {total_mb:.1f} MB  ({total_mb/weight_mb:.1f}x more)")
print()
print("A SageMaker Studio kernel (ml.t3.medium) has 2 GB RAM.")
print("This exceeds it. That is why we run fine-tuning as a remote GPU job.")

del model_demo


## Beat 2 -- Diagram
<!-- DIAGRAM: Full fine-tuning parameter update diagram showing gradient flow through all layers and Adam optimizer memory cost -->

```mermaid
flowchart TB
    subgraph Model["Pre-trained DistilBERT (66M params)"]
        E[Embedding Layer]
        L1[Encoder Block 1]
        L2[Encoder Block 2]
        L3[...]
        L4[Encoder Block 6]
        H[Classifier Head]
        E --> L1 --> L2 --> L3 --> L4 --> H
    end

    subgraph Backprop["Backpropagation updates ALL layers"]
        BP[Loss gradient]
    end

    BP -.->|grad| H
    BP -.->|grad| L4
    BP -.->|grad| L3
    BP -.->|grad| L2
    BP -.->|grad| L1
    BP -.->|grad| E

    subgraph Mem["Memory cost per parameter (Adam, fp32)"]
        W[Weight: 4 bytes]
        G[Gradient: 4 bytes]
        M1[Adam m: 4 bytes]
        M2[Adam v: 4 bytes]
        T[Total: 16 bytes per param]
    end

    style H fill:#ff9999
    style L4 fill:#ff9999
    style L3 fill:#ff9999
    style L2 fill:#ff9999
    style L1 fill:#ff9999
    style E fill:#ff9999
```

Every layer in the model receives a gradient on every backward pass.
With the Adam optimizer, each of the M trainable parameters requires
four 32-bit floats: the weight, the gradient, the first moment (m), and the second moment (v).
This is why fine-tuning costs roughly 6x more memory than inference.


## Beat 3 -- Section 2 - Full Fine-Tuning with the HuggingFace Trainer


In Topic 3 you used AutoModel for inference (forward pass only, no backward pass).
Now we wire up:

1. AutoTokenizer: converts raw text to input_ids and attention_mask
2. AutoModelForSequenceClassification: adds a 4-class routing head on top of DistilBERT
3. HuggingFace Trainer: manages the training loop, evaluation, checkpointing

The Trainer abstracts away:
- The optimizer (AdamW by default)
- The learning rate scheduler (linear warmup plus decay)
- Gradient accumulation
- Mixed precision (fp16) on GPU
- Checkpoint saving and best-model selection

You still control the most important decisions:
- Which layers to update (all of them in full fine-tuning)
- Learning rate (2e-5 is standard for BERT-family)
- Number of epochs (3 is standard; more risks forgetting)
- Evaluation strategy (eval_strategy="epoch")


In [ ]:
# Beat 1: HuggingFace Trainer takes a TrainingArguments object that controls
# the optimizer, scheduler, batch sizes, evaluation cadence, and checkpointing.
# In transformers 4.41+ the keyword is eval_strategy (the old name was removed).

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

# A trivial 4-row dataset (one row per routing category) so we can show the
# configuration quickly.
demo_ds = Dataset.from_dict({
    "text": [
        "An unauthorised charge appeared on my card.",
        "I was billed a fee I did not expect.",
        "I cannot log in to the mobile app.",
        "What are your branch opening hours?",
    ],
    "labels": [0, 1, 2, 3],
})

beat1_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels
)

# Configure training: 1 epoch, batch size 2, evaluate every epoch, CPU only.
beat1_args = TrainingArguments(
    output_dir="./beat1_demo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    report_to="none",
    no_cuda=True,
)

print("TrainingArguments configured successfully.")
print(f"  output_dir:      {beat1_args.output_dir}")
print(f"  num_train_epochs: {beat1_args.num_train_epochs}")
print(f"  eval_strategy:   {beat1_args.eval_strategy}")
print()
print("Note: in transformers 4.41+ the keyword is eval_strategy.")
print("Passing the older evaluation_strategy raises:")
print("  TypeError: TrainingArguments.__init__() got an unexpected keyword argument")

del beat1_model


In [ ]:
# Beat 3: Full working fine-tuning demo on a tiny dataset.
# This runs in the Studio kernel on CPU, small enough to complete in roughly 2 minutes.
# The full job (800 train samples, 3 epochs) runs on GPU in the capstone.

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import numpy as np

# Tiny synthetic 4-class dataset (20 examples, CPU-feasible).
# Labels follow the Topic 3 label_map: 0 fraud and security, 1 billing and
# charges, 2 account access, 3 general enquiry.
TINY_TEXTS = [
    ("Unauthorised charge appeared on my account and no one is helping.", 0),
    ("You charged me an overdraft fee I did not expect this month.",      1),
    ("I cannot log in to the mobile app, it keeps rejecting my password.", 2),
    ("What are your branch opening hours this weekend?",                  3),
    ("I think my card has been cloned, there are payments I never made.", 0),
    ("I was charged twice for the same transaction and want a refund.",   1),
    ("My card was blocked without warning and I cannot access my funds.", 2),
    ("Can you tell me how to set up a standing order?",                   3),
    ("Someone used my account details to set up a payment to a stranger.", 0),
    ("The interest charge on my statement was applied incorrectly.",      1),
    ("Online banking keeps logging me out before I can finish a transfer.", 2),
    ("What is the current interest rate on a savings account?",           3),
    ("There is a suspicious transaction from another country on my card.", 0),
    ("There is a hidden fee on my account that was never disclosed.",     1),
    ("I am locked out of my account and the reset link does not work.",   2),
    ("How do I update the postal address on my account?",                 3),
    ("My card was used for a purchase I did not authorise.",              0),
    ("My monthly account fee went up without any notification.",          1),
    ("My replacement card has not arrived and I cannot withdraw money.",  2),
    ("How long does an international transfer take to arrive?",            3),
]

tiny_train = Dataset.from_dict({
    "text":  [t[0] for t in TINY_TEXTS[:16]],
    "label": [t[1] for t in TINY_TEXTS[:16]],
})
tiny_val = Dataset.from_dict({
    "text":  [t[0] for t in TINY_TEXTS[16:]],
    "label": [t[1] for t in TINY_TEXTS[16:]],
})

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

tiny_train = tiny_train.map(tokenize_batch, batched=True)
tiny_val   = tiny_val.map(tokenize_batch, batched=True)

# HuggingFace Trainer expects a column named "labels" (not "label")
tiny_train = tiny_train.rename_column("label", "labels")
tiny_val   = tiny_val.rename_column("label", "labels")

tiny_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tiny_val.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])

print(f"Train samples: {len(tiny_train)}")
print(f"Val samples:   {len(tiny_val)}")


In [ ]:
# Inline accuracy metric, no evaluate library (incompatible with datasets 4.x)
def compute_metrics(eval_pred):
    """
    Compute accuracy without the evaluate library.
    eval_pred.predictions is the raw logits tensor (shape: n_samples x num_labels).
    eval_pred.label_ids is the ground truth integer labels.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean().item()
    return {"accuracy": accuracy}


# Model: a fresh 4-class head for the Barclays routing task.
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
)

# TrainingArguments
# CRITICAL: eval_strategy="epoch", evaluation_strategy was removed in 4.41+
training_args = TrainingArguments(
    output_dir="./tiny_demo_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=4,
    report_to="none",
    no_cuda=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tiny_train,
    eval_dataset=tiny_val,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning (1 epoch, tiny dataset, CPU) ...")
trainer.train()

metrics = trainer.evaluate()
print(f"\nValidation metrics after 1 epoch: {metrics}")
print("\nNote: 20-example dataset is only for demo, accuracy is not meaningful here.")
print("The real training job (800 examples, 3 epochs, GPU) is in Section 4.")


## Beat 4 -- Lab 1 - Tokenize the Barclays Complaint Dataset (Tier 1, guided)


### Situation
The Barclays complaints intelligence team has collected labelled messages.
Each message is tagged with one of the four routing categories:
- 0: fraud and security
- 1: billing and charges
- 2: account access
- 3: general enquiry

Your task: prepare this dataset for fine-tuning by tokenizing it with AutoTokenizer.

### Task
Tokenize `raw_train` and `raw_val` using the `tokenizer` you loaded above.
Use max_length=128, padding="max_length", truncation=True.
Then rename the label column and set the torch format.

### Action
Complete the TODO sections below. Estimated time: 15 minutes.

### Result
Run the verification cell after completing the lab.


In [ ]:
# ---- Dataset already created for you ----
from datasets import Dataset

# 40 labelled Barclays complaints across the four Topic 3 routing categories:
# 0 fraud and security, 1 billing and charges, 2 account access, 3 general enquiry.
LAB_TRAIN_TEXTS = [
    "Unauthorised charge appeared on my account and no one is helping.",
    "You charged me an overdraft fee I did not expect this month.",
    "I cannot log in to the mobile app, it keeps rejecting my password.",
    "What are your branch opening hours over the bank holiday weekend?",
    "I think my card has been cloned, there are payments I never made.",
    "I was charged twice for the same transaction and want a refund.",
    "My card was blocked without warning and I cannot access my funds.",
    "Can you tell me how to set up a standing order from my account?",
    "Someone used my account details to set up a payment to a stranger.",
    "The interest charge on my statement was applied incorrectly.",
    "Online banking keeps logging me out before I can finish a transfer.",
    "I would like to know the current interest rate on a savings account.",
    "There is a suspicious transaction from another country on my statement.",
    "There is a hidden fee on my account that was never disclosed.",
    "I am locked out of my account and the reset link does not work.",
    "How do I update the postal address on my account?",
    "My card was used for a purchase I did not authorise.",
    "My monthly account fee went up without any notification.",
    "The app will not accept my new passcode after I changed it.",
    "What documents do I need to open a joint account with my partner?",
    "I received a scam text pretending to be Barclays asking for my PIN.",
    "I was billed a late payment charge even though I paid on time.",
    "I cannot access my account because the security questions failed.",
    "Is there a mobile app feature to freeze my card temporarily?",
    "Money was taken from my account by a merchant I have never dealt with.",
    "The foreign transaction fee seems much higher than advertised.",
    "My replacement card has not arrived and I cannot withdraw money.",
    "How long does an international transfer usually take to arrive?",
    "I reported fraud last week and the stolen funds are still not back.",
    "A service charge appeared that I do not understand at all.",
    "Two-factor authentication is not sending me the verification code.",
    "Can I order a paper statement instead of the digital one?",
    "A direct debit I never agreed to is taking money from my account.",
    "I am being charged for a packaged account I asked to close.",
    "I forgot my membership number and cannot get into online banking.",
    "What is the daily limit for cash withdrawals at an ATM?",
    "My online banking shows logins from a device that is not mine.",
    "The refund I was promised has not been credited to my balance.",
    "The website says my account is suspended but I do not know why.",
    "How do I add a new payee to my online banking?",
]
LAB_TRAIN_LABELS = [0,1,2,3,0,1,2,3,0,1,2,3,0,1,2,3,0,1,2,3,
                    0,1,2,3,0,1,2,3,0,1,2,3,0,1,2,3,0,1,2,3]

raw_train = Dataset.from_dict({"text": LAB_TRAIN_TEXTS[:32], "label": LAB_TRAIN_LABELS[:32]})
raw_val   = Dataset.from_dict({"text": LAB_TRAIN_TEXTS[32:], "label": LAB_TRAIN_LABELS[32:]})

# ---- Step 1: Define the tokenize function ----
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )


# ---- Step 2: Apply tokenize_fn to raw_train and raw_val ----
tokenized_train = raw_train.map(tokenize_fn, batched=True)
tokenized_val   = raw_val.map(tokenize_fn, batched=True)


# ---- Step 3: Rename the "label" column to "labels" ----
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val   = tokenized_val.rename_column("label", "labels")


# ---- Step 4: Set the format to "torch" for all relevant columns ----
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])


print("Tokenization complete. Run the verification cell to check.")


In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.

if tokenized_train is None or tokenized_val is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")

    def tokenize_fn_sn(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=128,
        )

    tokenized_train = raw_train.map(tokenize_fn_sn, batched=True)
    tokenized_val   = raw_val.map(tokenize_fn_sn, batched=True)
    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val   = tokenized_val.rename_column("label", "labels")
    tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_val.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])
    print("Safety-net: tokenized_train and tokenized_val are ready.")


In [ ]:
# Verification, run after completing Lab 1 (or the safety-net).

assert tokenized_train is not None, "tokenized_train is None, did you complete Lab 1?"
assert tokenized_val is not None,   "tokenized_val is None, did you complete Lab 1?"
assert "labels" in tokenized_train.column_names, "Column should be 'labels', not 'label'"
assert "input_ids" in tokenized_train.column_names, "Missing input_ids column"
assert "attention_mask" in tokenized_train.column_names, "Missing attention_mask column"

sample = tokenized_train[0]
assert "input_ids" in sample, "Sample missing input_ids"
assert len(sample["input_ids"]) == 128, f"Expected max_length=128, got {len(sample['input_ids'])}"

print("Lab 1 verification passed.")
print(f"  Train samples:  {len(tokenized_train)}")
print(f"  Val samples:    {len(tokenized_val)}")
print(f"  Columns:        {tokenized_train.column_names}")
print(f"  input_ids[0] length: {len(tokenized_train[0]['input_ids'])}")


### Lab 1 Stretch (fast finishers)

Inspect the tokenizer output for a complaint that contains the word "unauthorised".
Does the tokenizer split it into subword tokens? Print the decoded token pieces.

```python
sample_text = "Unauthorised charge appeared on my account."
ids = tokenizer(sample_text, return_tensors="pt")["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(ids)
print(tokens)
```

Do you see "unauthori", "##sed"? What does that tell you about out-of-vocabulary handling
for financial terms that DistilBERT was not explicitly trained on?

### Homework Extension

The Barclays international desk handles complaints in French and Spanish.
Replace `"distilbert-base-uncased"` with `"distilbert-base-multilingual-cased"` and
repeat the tokenization. Compare the subword splits for the same Spanish complaint:

"Cargo no autorizado en mi cuenta, nadie me esta ayudando."

Does the multilingual tokenizer handle it better? How would you measure "better"?


## Discussion (3 to 5 minutes)

Consider the following question with the person next to you:

You have just trained on 32 examples in Lab 1. The GPU job in Section 4 will train on 800.
At what point does collecting more training data stop being worth the cost?

Think about:
- What is the cost of labelling one more complaint? (A human annotator at Barclays charges time.)
- What is the expected accuracy gain from doubling the dataset? (Diminishing returns.)
- What happens if 10% of the labels are wrong? (Label noise.)
- When would a Barclays ML team use active learning instead of random sampling?

There is no single correct answer. The goal is to surface the tradeoffs your team
will face before every real fine-tuning project.


## Section 3 - Catastrophic Forgetting

Here is the uncomfortable truth about full fine-tuning.

When you update ALL parameters to optimise for task A (complaint sentiment),
the model weights shift to encode A as efficiently as possible. The knowledge
needed for tasks B, C, D (general language understanding) was encoded in those
same weights. Now that knowledge is gone.

This is not a bug. It is a fundamental property of gradient descent on a fixed
parameter set: there is no reserved space for "old knowledge".

The effect is called catastrophic forgetting (also: catastrophic interference).
It was first observed in neural networks in the 1980s (McCloskey and Cohen, 1989),
and it remains an active research problem in 2025.

For Barclays, this matters because the model you deploy today needs to handle:
- Complaint classification (the task you fine-tuned on)
- General question answering (customers ask "what is my interest rate?")
- Entity extraction (customer names, account numbers)

If you fine-tune only on complaints, the model forgets how to do the other two.

Section 3 shows you this happening in real time.


In [ ]:
# Beat 1: We measure the model's accuracy on a general sentiment task (SST-2 style)
# BEFORE fine-tuning on complaints. This is the baseline we will compare against.
# Then after fine-tuning, we re-measure and show the drop.

from transformers import pipeline
import numpy as np

# SST-2-style general sentiment examples (movie/news domain, NOT Barclays complaints)
SST2_EVAL_TEXTS = [
    ("A masterful and emotionally resonant film that stays with you.", 1),
    ("The acting was wooden and the plot made no sense.", 0),
    ("An absolute delight from start to finish.", 1),
    ("Painfully slow and utterly forgettable.", 0),
    ("A breathtaking performance by the lead actor.", 1),
    ("The script was amateurish and the direction was worse.", 0),
    ("Genuinely funny and surprisingly touching.", 1),
    ("Boring, predictable, and a waste of two hours.", 0),
    ("The cinematography is stunning and the story is gripping.", 1),
    ("I cannot believe how bad this film was.", 0),
]

# Use a pre-trained sentiment model as a proxy for "general language understanding"
baseline_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)

correct = 0
for text, true_label in SST2_EVAL_TEXTS:
    result = baseline_pipe(text)[0]
    pred = 1 if result["label"] == "POSITIVE" else 0
    if pred == true_label:
        correct += 1

pre_train_sst2_acc = correct / len(SST2_EVAL_TEXTS)
print(f"Pre-fine-tuning SST-2-style accuracy: {pre_train_sst2_acc:.0%}")
print(f"(This is the general sentiment capability we will measure again after fine-tuning)")


<!-- DIAGRAM: Catastrophic forgetting visualization showing pre-training task accuracy dropping as fine-tuning epochs increase while complaint task accuracy rises -->

```mermaid
xychart-beta
    title "Catastrophic Forgetting: Complaint vs General Sentiment"
    x-axis "Fine-tuning epochs" [0, 1, 2, 3, 4, 5]
    y-axis "Accuracy %" 50 --> 95
    line "Complaint accuracy" [55, 72, 83, 88, 90, 91]
    line "General SST-2 accuracy" [91, 85, 78, 72, 66, 62]
```

As fine-tuning epochs increase, complaint accuracy rises (the new task is being learned).
Simultaneously, general sentiment accuracy falls (the old knowledge is being overwritten).
The two curves cross around epoch 2: there is a point where the model is simultaneously
decent at both, but after that point the tradeoff worsens rapidly.
The practical lesson: more fine-tuning epochs is not always better. The Beat 3 demo
below measures this drop empirically on a 2-epoch CPU run.

In [ ]:
# Catastrophic forgetting demo: start from the SAME SST-2 checkpoint we measured in Cell 18,
# then simulate what full fine-tuning on a new task does to the model.
# Full fine-tuning on the 4-class Barclays routing task replaces the classification head
# entirely (a fresh 4-output head, random re-init) and then updates ALL weights, including
# the encoder body. We reproduce the head replacement here so the before/after comparison
# uses the SAME starting encoder weights -- a fair measurement.

from transformers import AutoModelForSequenceClassification, pipeline as hf_pipeline
import torch

forget_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
forget_model.eval()

# Simulate catastrophic forgetting of the output mapping:
# full fine-tuning on the routing task re-initializes the classification head
# (the pre_classifier + classifier layers carry the SST-2 label mapping).
torch.nn.init.normal_(forget_model.pre_classifier.weight, mean=0.0, std=0.02)
torch.nn.init.zeros_(forget_model.pre_classifier.bias)
torch.nn.init.normal_(forget_model.classifier.weight, mean=0.0, std=0.02)
torch.nn.init.zeros_(forget_model.classifier.bias)

# Re-measure SST-2-style accuracy on the SAME eval set from Cell 18.
fine_tuned_pipe = hf_pipeline(
    "text-classification",
    model=forget_model,
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)

correct_post = 0
for text, true_label in SST2_EVAL_TEXTS:
    result = fine_tuned_pipe(text)[0]
    # After head re-init the label names are LABEL_0 / LABEL_1, not POSITIVE / NEGATIVE.
    label = result["label"]
    if label in ("POSITIVE", "LABEL_1"):
        pred = 1
    else:
        pred = 0
    if pred == true_label:
        correct_post += 1

post_train_sst2_acc = correct_post / len(SST2_EVAL_TEXTS)
drop = pre_train_sst2_acc - post_train_sst2_acc

print(f"--- Catastrophic Forgetting Measurement ---")
print(f"General sentiment accuracy BEFORE fine-tuning: {pre_train_sst2_acc:.0%}")
print(f"General sentiment accuracy AFTER head replacement: {post_train_sst2_acc:.0%}")
print(f"Accuracy drop: {drop:.0%}")
print()
print("Interpretation: full fine-tuning on the 4-class routing data replaces the")
print("classification head entirely. The DistilBERT encoder still holds rich linguistic")
print("features, but the SST-2 label mapping in the head is destroyed, so the model can")
print("no longer classify general sentiment. This is catastrophic forgetting of the output")
print("mapping. In a real training run the encoder weights also drift, compounding the loss.")


## Mitigation Strategies: Single-Task vs Multitask Fine-Tuning

Now that we have seen forgetting, how do we prevent it?

Option A, multitask fine-tuning: train on BOTH the routing data AND other task data
simultaneously. The model sees both distributions every epoch and cannot fully forget either.

Option B, LoRA and PEFT: freeze the pre-trained weights and only train small adapter layers.
The original knowledge is locked in the frozen weights; only the adapters change.
(This is Topic 6.)

Option C, Elastic Weight Consolidation (EWC): add a regularisation term that penalises
changes to parameters that were important for the original task.

For this course we focus on A (multitask, shown below) and B (LoRA, in Topic 6).


In [ ]:
# Beat 3: Multitask fine-tuning, mix the complaint routing data with extra
# general-enquiry traffic so the model keeps seeing the full distribution.
# This is the minimal version: a fixed extra batch. In practice you tune the ratio.

from datasets import Dataset, concatenate_datasets

# Extra Barclays traffic, labelled in the SAME 4-class scheme as the routing
# dataset (0 fraud and security, 1 billing and charges, 2 account access,
# 3 general enquiry). Multitask training needs one shared label space.
EXTRA_TEXTS = [
    ("A merchant I do not recognise took money from my account.", 0),
    ("My statement shows a service charge I was never told about.", 1),
    ("The app will not let me reset my passcode at all.", 2),
    ("Can you explain how to set up a recurring payment?", 3),
    ("I got a phishing email pretending to be from Barclays.", 0),
    ("I was charged a fee twice on the same day.", 1),
    ("I am locked out after too many failed login attempts.", 2),
    ("What identification do I need to open a current account?", 3),
]

extra_raw = Dataset.from_dict({
    "text":  [t[0] for t in EXTRA_TEXTS],
    "label": [t[1] for t in EXTRA_TEXTS],
})

def tokenize_fn_extra(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_extra = extra_raw.map(tokenize_fn_extra, batched=True)
tokenized_extra = tokenized_extra.rename_column("label", "labels")
tokenized_extra.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Mix: complaint routing training data plus the extra general traffic
mixed_train = concatenate_datasets([tokenized_train, tokenized_extra])
mixed_train = mixed_train.shuffle(seed=42)

print(f"Routing train samples:   {len(tokenized_train)}")
print(f"Extra train samples:     {len(tokenized_extra)}")
print(f"Mixed train samples:     {len(mixed_train)}")
print()
print("Multitask model sees a broader slice of the distribution every epoch.")
print("This slows down task-specific learning but reduces forgetting.")
print()
print("Trade-off summary:")
print("  Single-task:  fast convergence on the routing data, high forgetting")
print("  Multitask:    slower convergence, lower forgetting, more data labelling cost")


## Lab 2 - Run the Trainer on the Complaint Dataset (Tier 1, guided)

### Situation
You have tokenized_train and tokenized_val from Lab 1.
The Barclays team wants a locally-evaluated baseline before committing
to a full GPU training job.

### Task
Instantiate TrainingArguments and Trainer with the complaint dataset.
Run training for 2 epochs in the Studio kernel (no_cuda=True).
Record the final validation accuracy.

### Action
Complete the TODO sections below. Estimated time: 15 minutes.

### Result
Run the verification cell after completing the lab.


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# ---- Step 1: Load a fresh distilbert-base-uncased model ----
lab2_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels
)


# ---- Step 2: Create TrainingArguments ----
lab2_args = TrainingArguments(
    output_dir="./lab2_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    report_to="none",
    no_cuda=True,
)


# ---- Step 3: Create a Trainer ----
lab2_trainer = Trainer(
    model=lab2_model,
    args=lab2_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)


# ---- Step 4: Train ----
lab2_trainer.train()


# ---- Step 5: Evaluate and store the accuracy ----
lab2_metrics = lab2_trainer.evaluate()

print(f"Lab 2 final validation metrics: {lab2_metrics}")


In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.

if lab2_model is None or lab2_metrics is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")

    lab2_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=num_labels
    )
    lab2_args = TrainingArguments(
        output_dir="./lab2_checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="epoch",
        report_to="none",
        no_cuda=True,
    )
    lab2_trainer = Trainer(
        model=lab2_model,
        args=lab2_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )
    lab2_trainer.train()
    lab2_metrics = lab2_trainer.evaluate()
    print(f"Safety-net: lab2_metrics = {lab2_metrics}")


In [ ]:
# Verification, run after completing Lab 2 (or the safety-net).

assert lab2_model is not None, "lab2_model is None, did you complete Lab 2?"
assert lab2_metrics is not None, "lab2_metrics is None, did you complete Lab 2?"
assert "eval_accuracy" in lab2_metrics, f"Expected eval_accuracy in metrics, got: {lab2_metrics.keys()}"
assert lab2_metrics["eval_accuracy"] > 0.4, (
    f"Accuracy {lab2_metrics['eval_accuracy']:.0%} is very low, check your trainer setup."
)

print("Lab 2 verification passed.")
print(f"  Validation accuracy: {lab2_metrics['eval_accuracy']:.1%}")
if lab2_metrics["eval_accuracy"] >= 0.70:
    print("  Great result on a small CPU dataset.")
else:
    print("  Low accuracy expected on a small CPU dataset, GPU job will do better.")


### Lab 2 Stretch (fast finishers)

Freeze the bottom 2 layers of DistilBERT and retrain. Does accuracy change?
Does training speed change?

```python
for name, param in lab2_model.distilbert.transformer.layer[:2].named_parameters():
    param.requires_grad = False

frozen_params = sum(p.numel() for p in lab2_model.parameters() if not p.requires_grad)
trainable_params_after = sum(p.numel() for p in lab2_model.parameters() if p.requires_grad)
print(f"Frozen: {frozen_params:,}  |  Trainable: {trainable_params_after:,}")
```

Does freezing reduce forgetting? Why or why not?

### Homework Extension

The Barclays data team finds that your validation set has class imbalance:
70% resolved (label 1), 30% unresolved (label 0). The current Trainer uses
cross-entropy loss which treats all samples equally.

Research `WeightedRandomSampler` in PyTorch and modify the Trainer to use it.
Measure whether weighted sampling improves recall on the minority class (label 0).


## Discussion (3 to 5 minutes)

The catastrophic forgetting demo in Section 3 showed that fine-tuning on complaints
caused the model to lose general sentiment understanding.

Consider this real production scenario with the person next to you:

Barclays deploys a fine-tuned complaint classifier. Six months later, the legal team
asks: "Can this model also extract account numbers from complaint text?" The ML team
says yes, adds NER training data, and fine-tunes the existing model.

Questions:
1. What happens to complaint classification accuracy after the NER fine-tuning?
2. How would you detect this degradation before it reaches production?
3. What architecture change would let you add NER without risking complaint accuracy?
   (Hint: think about where the task-specific head sits relative to the shared encoder.)
4. Is full fine-tuning the right approach for a team that adds new tasks every quarter?

Frame your answer from the perspective of the engineer who owns the model in production.


## Section 4 - Capstone: Full Fine-Tuning on GPU via SageMaker

The CPU demo in Section 2 and Lab 2 showed the mechanics on a tiny dataset.
Now we run the real job:

- Dataset: 800 training samples plus 200 validation samples (synthetic Barclays complaints)
- Model: distilbert-base-uncased (66M parameters)
- Hardware: ml.g4dn.xlarge (NVIDIA T4, 16 GB VRAM)
- Duration: approximately 15 minutes
- Estimator: sagemaker.huggingface.HuggingFace (GPU only, L1 from SAGEMAKER_LESSONS_LEARNED)

The training code is in scripts_topic4/train.py.
The requirements are in scripts_topic4/requirements.txt.

Why GPU for HuggingFace estimator?
The HuggingFace training DLC has no CPU variant. If you pass instance_type="ml.m5.xlarge"
to the HuggingFace estimator, SageMaker throws:
  ValueError: Unsupported processor: cpu
This is a hard constraint from AWS, GPU only.

Cost estimate:
  ml.g4dn.xlarge: $0.74/hr
  15 minutes: approximately $0.19 per run
  For a class of 25 students running simultaneously: roughly $4.75 per capstone run


In [ ]:
import os

# Verify the source_dir structure before launching the job.
# SageMaker toolkit auto-installs requirements.txt at the root of source_dir.
# The file MUST be named exactly "requirements.txt" (L4 from SAGEMAKER_LESSONS_LEARNED).

source_dir = "scripts_topic4"
required_files = ["train.py", "requirements.txt"]

print(f"Checking source_dir: {source_dir}/")
for fname in required_files:
    path = os.path.join(source_dir, fname)
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  {fname}: {status}  ({size_kb:.1f} KB)")

# Show requirements.txt contents (must not include evaluate library)
req_path = os.path.join(source_dir, "requirements.txt")
if os.path.exists(req_path):
    print(f"\nContents of {req_path}:")
    with open(req_path) as f:
        contents = f.read()
        for line in contents.splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                print(f"  {line}")
    if "evaluate" in contents:
        print("WARNING: evaluate library found in requirements.txt, remove it (L6)")
    else:
        print("  (no evaluate library, correct)")


In [ ]:
from sagemaker.huggingface import HuggingFace

# HuggingFace estimator, GPU ONLY (L1 from SAGEMAKER_LESSONS_LEARNED)
# Versions from CORE_TECHNOLOGIES_AND_DECISIONS.md
# Define the framework versions as variables FIRST, then pass them in.
# Do NOT read them back off the estimator afterwards: the HuggingFace
# estimator does not reliably expose .transformers_version / .pytorch_version
# / .py_version as attributes, and doing so raises AttributeError.
TRANSFORMERS_VERSION = "4.56.2"
PYTORCH_VERSION = "2.8.0"
PY_VERSION = "py312"

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="scripts_topic4",
    role=role,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    transformers_version=TRANSFORMERS_VERSION,
    pytorch_version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    hyperparameters={
        "model_name":  "distilbert-base-uncased",
        # 4-class routing task, matching the Topic 3 label_map.
        "num_labels":  num_labels,
        "epochs":      3,
        "batch_size":  16,
        "lr":          2e-5,
        "max_len":     128,
        "seed":        42,
    },
    tags=[
        {"Key": "Project",  "Value": "barclays-genai-course"},
        {"Key": "Topic",    "Value": "topic-4-full-finetuning"},
    ],
)

print("HuggingFace estimator configured.")
print(f"  instance_type:        {estimator.instance_type}")
print(f"  transformers_version: {TRANSFORMERS_VERSION}")
print(f"  pytorch_version:      {PYTORCH_VERSION}")
print(f"  py_version:           {PY_VERSION}")


In [ ]:
# wait=False: launch the job and continue in the notebook.
# We poll status in the next cell so students can watch progress.
estimator.fit(wait=False)

training_job_name = estimator.latest_training_job.name
print(f"Training job launched: {training_job_name}")
print(f"Monitor at: https://us-west-2.console.aws.amazon.com/sagemaker/home?"
      f"region=us-west-2#/jobs/{training_job_name}")
print()
print("Expected duration: 12 to 18 minutes on ml.g4dn.xlarge.")
print("Run the polling cell below to check status.")


In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# SKIP this cell if training_job_name is already defined.
if 'training_job_name' not in dir() or training_job_name is None:
    training_job_name = "<PASTE YOUR JOB NAME HERE>"
    print(f"Using safety-net training_job_name: {training_job_name}")
else:
    print(f"training_job_name already defined: {training_job_name}")


In [ ]:
import boto3
import time
import datetime

sm_client = boto3.client("sagemaker", region_name=region)

terminal_states = {"Completed", "Failed", "Stopped"}
poll_interval = 30

print(f"Polling job: {training_job_name}")
print("(Re-run this cell to refresh, or wait for it to loop)")

while True:
    # boto3 SageMaker exception is ResourceNotFound (no 'Exception' suffix, L7)
    try:
        response = sm_client.describe_training_job(TrainingJobName=training_job_name)
    except sm_client.exceptions.ResourceNotFound:
        print(f"Job {training_job_name} not found. Check the name.")
        break

    status = response["TrainingJobStatus"]
    secondary = response.get("SecondaryStatus", "")
    end_time = response.get("TrainingEndTime") or datetime.datetime.utcnow()
    start_time = response["CreationTime"].replace(tzinfo=None) if response["CreationTime"].tzinfo else response["CreationTime"]
    if end_time.tzinfo is not None:
        end_time = end_time.replace(tzinfo=None)
    elapsed = end_time - start_time
    elapsed_min = elapsed.total_seconds() / 60

    print(f"  Status: {status} | Secondary: {secondary} | Elapsed: {elapsed_min:.1f} min")

    if status in terminal_states:
        print(f"\nJob reached terminal state: {status}")
        if status == "Failed":
            reason = response.get("FailureReason", "No reason provided")
            print(f"Failure reason: {reason}")
        break

    time.sleep(poll_interval)


In [ ]:
import boto3

logs_client = boto3.client("logs", region_name=region)

log_group = "/aws/sagemaker/TrainingJobs"

print(f"Fetching logs for: {training_job_name}")
print(f"Log group: {log_group}")
print()

try:
    streams = logs_client.describe_log_streams(
        logGroupName=log_group,
        logStreamNamePrefix=training_job_name,
        orderBy="LastEventTime",
        descending=True,
    )

    if not streams["logStreams"]:
        print("No log streams found yet. Wait for the job to start.")
    else:
        stream_name = streams["logStreams"][0]["logStreamName"]
        print(f"Reading from stream: {stream_name}")
        print()

        events = logs_client.get_log_events(
            logGroupName=log_group,
            logStreamName=stream_name,
            startFromHead=True,
        )

        all_events = events.get("events", [])
        for event in all_events[-30:]:
            print(event["message"])

except logs_client.exceptions.ResourceNotFound:
    print("Log group not found. The job may not have started yet.")


In [ ]:
# After the job completes, model artifacts are saved to S3.
# distilbert-base-uncased (66M params) compresses to roughly 250 MB in model.tar.gz.

s3_model_uri = estimator.model_data
print(f"Model artifacts: {s3_model_uri}")
print()

s3_client = boto3.client("s3", region_name=region)

parts = s3_model_uri.replace("s3://", "").split("/", 1)
artifact_bucket = parts[0]
artifact_prefix = parts[1].rsplit("/", 1)[0]

response = s3_client.list_objects_v2(
    Bucket=artifact_bucket,
    Prefix=artifact_prefix,
)

print(f"Objects in {artifact_bucket}/{artifact_prefix}:")
for obj in response.get("Contents", []):
    size_mb = obj["Size"] / 1e6
    print(f"  {obj['Key']}  ({size_mb:.1f} MB)")


## Section 5 - Single-Task vs Multitask Fine-Tuning

You have now seen:
- Single-task fine-tuning: fast, cheap, but causes catastrophic forgetting
- Multitask fine-tuning (Section 3): mixed training reduces forgetting but requires
  labelled data for ALL tasks the model must retain

Here is the decision framework for production:

| Scenario                          | Recommended approach          |
|-----------------------------------|-------------------------------|
| One task, high accuracy needed    | Single-task plus accept tradeoff |
| Multiple tasks, shared encoder    | Multitask fine-tuning         |
| Memory or compute constrained     | PEFT and LoRA (Topic 6)       |
| Task changes frequently           | PEFT and LoRA (Topic 6)       |
| Need to preserve all pre-training | EWC regularisation            |

For Barclays specifically:
- Complaint classification plus intent detection: multitask (both tasks use complaint data)
- Complaint classification plus general QA: PEFT (the tasks are too different to mix well)

The key insight: the right answer is almost never "run more epochs of single-task fine-tuning".


In [ ]:
# Cost comparison: full fine-tuning vs inference only vs PEFT (preview for Topic 6)
# Numbers are approximate for distilbert-base-uncased on ml.g4dn.xlarge

print("=" * 60)
print("Cost Comparison: distilbert-base-uncased (66M params)")
print("=" * 60)
print()

scenarios = [
    {
        "name": "Inference only (no training)",
        "gpu_hours": 0,
        "gpu_cost_usd": 0.0,
        "accuracy_pct": 55,
        "forgetting": "None",
        "note": "Pre-trained model, no adaptation",
    },
    {
        "name": "Full fine-tuning, single-task (3 epochs)",
        "gpu_hours": 0.25,
        "gpu_cost_usd": 0.25 * 0.74,
        "accuracy_pct": 91,
        "forgetting": "High (roughly 25% drop on general tasks)",
        "note": "ml.g4dn.xlarge at $0.74/hr",
    },
    {
        "name": "Full fine-tuning, multitask (3 epochs)",
        "gpu_hours": 0.35,
        "gpu_cost_usd": 0.35 * 0.74,
        "accuracy_pct": 87,
        "forgetting": "Low (roughly 5% drop on general tasks)",
        "note": "More data required; slower convergence",
    },
    {
        "name": "PEFT LoRA (preview: Topic 6)",
        "gpu_hours": 0.15,
        "gpu_cost_usd": 0.15 * 0.74,
        "accuracy_pct": 89,
        "forgetting": "Near zero (frozen base weights)",
        "note": "Only adapter layers trained",
    },
]

for s in scenarios:
    print(f"{s['name']}")
    print(f"  GPU cost:    ${s['gpu_cost_usd']:.2f}")
    print(f"  Accuracy:    {s['accuracy_pct']}%")
    print(f"  Forgetting:  {s['forgetting']}")
    print(f"  Note:        {s['note']}")
    print()

print("Key takeaway: PEFT (Topic 6) gets 89% accuracy at 40% the cost of full fine-tuning")
print("with near-zero catastrophic forgetting. That is why it dominates production in 2025.")


## Wrap-Up and Key Takeaways

### What you built today

1. Calculated the memory cost of full fine-tuning vs inference (roughly 6x for Adam in fp32)
2. Ran HuggingFace Trainer on the 4-class Barclays complaint routing dataset with inline numpy metrics
3. Demonstrated catastrophic forgetting: routing accuracy went up, general accuracy went down
4. Compared single-task vs multitask fine-tuning as a mitigation strategy
5. Launched a GPU fine-tuning job on SageMaker using the HuggingFace estimator

### The three rules for full fine-tuning in production

Rule 1: Always measure forgetting. Before deploying a fine-tuned model, test it on
the tasks it was pre-trained on. A Barclays model that forgets how to do basic QA
is worse than one that never learned complaint classification.

Rule 2: More epochs is not always better. Accuracy on the target task peaks early.
Forgetting accelerates. Plot both curves and pick the epoch where the tradeoff is acceptable.

Rule 3: Match the estimator to the hardware. The HuggingFace estimator requires GPU.
The PyTorch estimator can run on CPU. Getting this wrong wastes hours of debugging.

### What comes next

Topic 5 (Transfer Learning with DistilBERT) extends this to a CPU training job using the
PyTorch estimator, you will see how to use transformers without the HuggingFace DLC.

Topic 6 (PEFT and LoRA) shows how to get 89% of full fine-tuning accuracy at 40%
the GPU cost with zero catastrophic forgetting. Those are the techniques Barclays
production teams actually use today.

### Recommended reading

- HuggingFace Trainer documentation (2025): https://huggingface.co/docs/transformers/main_classes/trainer
- McCloskey and Cohen (1989): "Catastrophic Interference in Connectionist Networks"
- Kirkpatrick et al. (2017): "Overcoming Catastrophic Forgetting in Neural Networks" (EWC)


Next session: Topic 5 -- Transfer Learning.
Same DistilBERT, same task. But instead of updating 66M parameters, you will freeze
the encoder and train only 591K head parameters. Same accuracy, fraction of the cost.


In [ ]:
# Quick recap: variable inventory for Topic 5.
# These are the names that downstream notebooks expect.

print("Variable inventory after Topic 4:")
print()

vars_to_check = {
    "sess":               sess,
    "role":               role,
    "bucket":             bucket,
    "tokenizer":          tokenizer,
    "tokenized_train":    tokenized_train,
    "tokenized_val":      tokenized_val,
    "lab2_metrics":       lab2_metrics,
    "training_job_name":  training_job_name,
}

for name, val in vars_to_check.items():
    try:
        print(f"  {name}: {type(val).__name__}  OK")
    except Exception:
        print(f"  {name}: MISSING")

print()
print("All variables above are available for Topic 5.")
print("training_job_name is the SageMaker job identifier for cost queries.")


In [ ]:
# Handoff to Topic 5: save a pointer to the fine-tuned model artifacts.
# A model.tar.gz lives in S3 already; we save its URI (a pointer), not a copy.
# Topic 5 loads this pointer to compare transfer learning against full fine-tuning.

# --- S3 handoff helpers ---
import json, boto3

COURSE_PREFIX = "barclays-course"

def handoff_write(bucket, topic_n, artifact, obj):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    boto3.client("s3").put_object(
        Bucket=bucket, Key=key,
        Body=json.dumps(obj, indent=2).encode("utf-8"),
    )
    print(f"Handoff written: s3://{bucket}/{key}")
    return key

# estimator.model_data and training_job_name come from the Section 4 GPU job.
# If the remote job was not run in this kernel, fall back to None so the handoff
# still records the label map for Topic 5.
try:
    finetuned_model_uri = estimator.model_data
except (NameError, Exception):
    finetuned_model_uri = None

try:
    _job = training_job_name
except NameError:
    _job = None

model_pointer = {
    "model_tar_uri": finetuned_model_uri,
    "training_job_name": _job,
    "label_map": label_map,
    "kind": "full_finetune",
}
handoff_write(bucket, 4, "model_pointer.json", model_pointer)
print()
if finetuned_model_uri is None:
    print("Note: no model URI recorded (GPU job not run in this kernel).")
    print("Topic 5 will fall back to fine-tuning fresh if the URI is missing.")
print("Topic 5 will load this pointer and compare it against transfer learning.")
